In [1]:

from pathlib import Path
from google.colab import drive
import pandas as pd
import torch
import json

PROJECT = Path("/content/drive5/MyDrive/Ocular_Project")

if not (PROJECT / "results/IITD").is_dir():
    drive.mount("/content/drive5")

if not (PROJECT / "results/IITD").is_dir():
    raise FileNotFoundError("IITD project folder not found.")

PATHS = {
    "baseline_split": PROJECT / "results/IITD/notebook04_baseline_recognition/IITD_baseline_dataset_split.csv",
    "baseline_metrics": PROJECT / "results/IITD/notebook04_baseline_recognition/IITD_baseline_test_metrics.csv",
    "baseline_checkpoint": PROJECT / "models/IITD/IITD_baseline_resnet18_best.pth",
    "traditional_manifest": PROJECT / "results/IITD/notebook09_dataset_augmentation/IITD_traditional_train_manifest.csv",
    "gan_manifest": PROJECT / "results/IITD/notebook09_dataset_augmentation/IITD_gan_manifest.csv",
}

OUT_DIR = PROJECT / "results/IITD/notebook10_recognition_augmented"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

for name, path in PATHS.items():
    print(f"{name}: {'FOUND' if path.is_file() else 'MISSING'}")

if PATHS["baseline_split"].is_file():
    split_df = pd.read_csv(PATHS["baseline_split"])
    print("\nSplit counts:")
    print(split_df["split"].value_counts())

if PATHS["traditional_manifest"].is_file():
    trad_df = pd.read_csv(PATHS["traditional_manifest"])
    print("\nTraditional augmentation records:", len(trad_df))
    print("Columns:", trad_df.columns.tolist())

if PATHS["gan_manifest"].is_file():
    gan_df = pd.read_csv(PATHS["gan_manifest"])
    print("\nGAN records:", len(gan_df))
    print("Columns:", gan_df.columns.tolist())

print("\nOutput directory:", OUT_DIR)
print("Cell 1 complete. No model training has started.")


Mounted at /content/drive5
PyTorch: 2.11.0+cpu
CUDA available: False
baseline_split: FOUND
baseline_metrics: FOUND
baseline_checkpoint: FOUND
traditional_manifest: FOUND
gan_manifest: FOUND

Split counts:
split
train         1526
test           436
validation     218
Name: count, dtype: int64

Traditional augmentation records: 4578
Columns: ['original_relative_path', 'augmented_local_path', 'filename', 'subject', 'eye', 'label_name', 'label', 'split', 'augmentation_id', 'augmentation_methods', 'data_source', 'identity_label_verified']

GAN records: 1275
Columns: ['synthetic_path', 'filename', 'label', 'label_name', 'sample_index', 'source_model', 'native_resolution', 'saved_resolution', 'identity_preservation_verified', 'split', 'data_source', 'identity_label_verified']

Output directory: /content/drive5/MyDrive/Ocular_Project/results/IITD/notebook10_recognition_augmented
Cell 1 complete. No model training has started.


In [2]:

from pathlib import Path
from zipfile import ZipFile
import pandas as pd
import json

# --------------------------------------------------
# 1. Load split and augmentation metadata
# --------------------------------------------------
split_df = pd.read_csv(PATHS["baseline_split"])
train_df = split_df[split_df["split"] == "train"].copy()
val_df = split_df[split_df["split"] == "validation"].copy()
test_df = split_df[split_df["split"] == "test"].copy()

trad_df = pd.read_csv(PATHS["traditional_manifest"])
gan_df = pd.read_csv(PATHS["gan_manifest"])

PREPROCESS_DIR = (
    PROJECT / "results/IITD/notebook02_preprocessing/images_224"
)
TRAD_ZIP = (
    PROJECT / "results/IITD/notebook05_traditional_augmentation/"
    "IITD_traditional_augmented_images.zip"
)

# --------------------------------------------------
# 2. Verify original images
# --------------------------------------------------
original_paths = []
missing_original = []

for row in train_df.itertuples(index=False):
    rel_png = Path(row.original_relative_path).with_suffix(".png")
    p = PREPROCESS_DIR / rel_png
    if p.is_file():
        original_paths.append(str(p))
    else:
        missing_original.append(str(p))

print("Original training images found:", len(original_paths))
print("Missing original images:", len(missing_original))

# --------------------------------------------------
# 3. Verify traditional archive members without extracting
# --------------------------------------------------
if not TRAD_ZIP.is_file():
    raise FileNotFoundError(f"Traditional augmentation ZIP missing: {TRAD_ZIP}")

with ZipFile(TRAD_ZIP) as zf:
    zip_names = set(zf.namelist())
    image_members = [
        n for n in zip_names
        if n.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
    ]

print("Traditional image members in ZIP:", len(image_members))
print("Traditional metadata records:", len(trad_df))

# --------------------------------------------------
# 4. Verify GAN paths
# --------------------------------------------------
gan_found = 0
gan_missing = []

for row in gan_df.itertuples(index=False):
    p = Path(row.synthetic_path)
    if not p.is_file():
        p = (
            PROJECT / "generated_images/IITD/gan"
            / str(row.label_name) / str(row.filename)
        )
    if p.is_file():
        gan_found += 1
    else:
        gan_missing.append(str(p))

print("GAN images found:", gan_found)
print("GAN images missing:", len(gan_missing))

# --------------------------------------------------
# 5. Save a verification report
# --------------------------------------------------
verification = {
    "original_train_expected": len(train_df),
    "original_train_found": len(original_paths),
    "original_missing": len(missing_original),
    "traditional_metadata_records": len(trad_df),
    "traditional_zip_images": len(image_members),
    "gan_metadata_records": len(gan_df),
    "gan_images_found": gan_found,
    "gan_images_missing": len(gan_missing),
    "validation_images": len(val_df),
    "test_images": len(test_df),
}

with open(OUT_DIR / "IITD_recognition_data_verification.json", "w") as f:
    json.dump(verification, f, indent=2)

print("\nVerification report:")
print(json.dumps(verification, indent=2))
print("\nCell 2 complete. No model training has started.")


Original training images found: 1526
Missing original images: 0
Traditional image members in ZIP: 4578
Traditional metadata records: 4578
GAN images found: 1275
GAN images missing: 0

Verification report:
{
  "original_train_expected": 1526,
  "original_train_found": 1526,
  "original_missing": 0,
  "traditional_metadata_records": 4578,
  "traditional_zip_images": 4578,
  "gan_metadata_records": 1275,
  "gan_images_found": 1275,
  "gan_images_missing": 0,
  "validation_images": 218,
  "test_images": 436
}

Cell 2 complete. No model training has started.


In [3]:

from pathlib import Path
import shutil
import zipfile
import json
import pandas as pd
from tqdm.auto import tqdm

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
PROJECT = Path("/content/drive5/MyDrive/Ocular_Project")

LOCAL_ROOT = Path("/content/IITD_notebook10_cache")
LOCAL_ORIGINAL = LOCAL_ROOT / "original_train"
LOCAL_TRADITIONAL = LOCAL_ROOT / "traditional_train"
LOCAL_GAN = LOCAL_ROOT / "gan_synthetic"

PREPROCESS_DIR = (
    PROJECT / "results/IITD/notebook02_preprocessing/images_224"
)

SPLIT_CSV = (
    PROJECT
    / "results/IITD/notebook04_baseline_recognition/"
      "IITD_baseline_dataset_split.csv"
)

TRAD_ZIP = (
    PROJECT
    / "results/IITD/notebook05_traditional_augmentation/"
      "IITD_traditional_augmented_images.zip"
)

TRAD_MANIFEST = (
    PROJECT
    / "results/IITD/notebook09_dataset_augmentation/"
      "IITD_traditional_train_manifest.csv"
)

GAN_MANIFEST = (
    PROJECT
    / "results/IITD/notebook09_dataset_augmentation/"
      "IITD_gan_manifest.csv"
)

# --------------------------------------------------
# 2. Read manifests
# --------------------------------------------------
split_df = pd.read_csv(SPLIT_CSV)
train_df = split_df[split_df["split"] == "train"].copy()

trad_df = pd.read_csv(TRAD_MANIFEST)
gan_df = pd.read_csv(GAN_MANIFEST)

# Avoid silently reusing a partial cache.
# This removes only our dedicated Notebook 10 cache.
if LOCAL_ROOT.exists():
    shutil.rmtree(LOCAL_ROOT)

LOCAL_ORIGINAL.mkdir(parents=True, exist_ok=True)
LOCAL_TRADITIONAL.mkdir(parents=True, exist_ok=True)
LOCAL_GAN.mkdir(parents=True, exist_ok=True)

# --------------------------------------------------
# 3. Copy original training images locally
# --------------------------------------------------
print("Copying original training images to local storage...")

original_records = []

for row in tqdm(train_df.itertuples(index=False), total=len(train_df)):
    relative_path = str(row.original_relative_path)

    # Use the processed 224x224 PNGs.
    source = PREPROCESS_DIR / Path(relative_path).with_suffix(".png")

    if not source.exists():
        raise FileNotFoundError(f"Missing original image: {source}")

    destination = LOCAL_ORIGINAL / Path(relative_path).with_suffix(".png")
    destination.parent.mkdir(parents=True, exist_ok=True)

    shutil.copy2(source, destination)

    original_records.append({
        "relative_path": relative_path,
        "local_path": str(destination),
        "label": int(row.label),
        "label_name": str(row.label_name)
    })

# --------------------------------------------------
# 4. Extract traditional augmented images locally
# --------------------------------------------------
print("\nExtracting traditional augmentation ZIP...")

with zipfile.ZipFile(TRAD_ZIP, "r") as archive:
    image_members = [
        name for name in archive.namelist()
        if name.lower().endswith((".png", ".jpg", ".jpeg", ".bmp"))
        and name.startswith("train/")
    ]

    if len(image_members) != len(trad_df):
        raise ValueError(
            f"ZIP/manifest mismatch: {len(image_members)} ZIP images "
            f"versus {len(trad_df)} manifest records."
        )

    for member in tqdm(image_members, desc="Extracting"):
        archive.extract(member, LOCAL_TRADITIONAL)

# --------------------------------------------------
# 5. Copy GAN images locally
# --------------------------------------------------
print("\nCopying GAN images to local storage...")

gan_records = []

for row in tqdm(gan_df.itertuples(index=False), total=len(gan_df)):
    source = Path(str(row.synthetic_path))

    if not source.exists():
        # Fallback for paths recorded under an older Drive mount.
        source = (
            PROJECT / "generated_images/IITD/gan"
            / str(row.label_name) / str(row.filename)
        )

    if not source.exists():
        raise FileNotFoundError(
            f"GAN image not found for {row.filename}: {source}"
        )

    destination = LOCAL_GAN / str(row.label_name) / str(row.filename)
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)

    gan_records.append({
        "local_path": str(destination),
        "label": int(row.label),
        "label_name": str(row.label_name),
        "identity_label_verified": bool(row.identity_label_verified)
    })

# --------------------------------------------------
# 6. Save local manifests and verify counts
# --------------------------------------------------
original_local_df = pd.DataFrame(original_records)
gan_local_df = pd.DataFrame(gan_records)

original_local_df.to_csv(
    LOCAL_ROOT / "original_train_local_manifest.csv", index=False
)
gan_local_df.to_csv(
    LOCAL_ROOT / "gan_local_manifest.csv", index=False
)

traditional_local_paths = [
    str(LOCAL_TRADITIONAL / member)
    for member in image_members
]

summary = {
    "original_training_images": len(original_records),
    "traditional_augmented_images": len(image_members),
    "gan_synthetic_images": len(gan_records),
    "original_missing": sum(
        not Path(p).exists() for p in original_local_df["local_path"]
    ),
    "gan_missing": sum(
        not Path(p).exists() for p in gan_local_df["local_path"]
    ),
    "traditional_missing": sum(
        not Path(p).exists() for p in traditional_local_paths
    ),
    "local_cache": str(LOCAL_ROOT),
    "note": (
        "GAN labels are conditioning labels and are not verified identities."
    )
}

with open(LOCAL_ROOT / "local_cache_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nLOCAL CACHE SUMMARY")
print(json.dumps(summary, indent=2))

assert summary["original_training_images"] == 1526
assert summary["traditional_augmented_images"] == 4578
assert summary["gan_synthetic_images"] == 1275
assert summary["original_missing"] == 0
assert summary["traditional_missing"] == 0
assert summary["gan_missing"] == 0

print("\nCell 3 complete. All verified training images are now local.")


Copying original training images to local storage...


  0%|          | 0/1526 [00:00<?, ?it/s]


Extracting traditional augmentation ZIP...


Extracting:   0%|          | 0/4578 [00:00<?, ?it/s]


Copying GAN images to local storage...


  0%|          | 0/1275 [00:00<?, ?it/s]


LOCAL CACHE SUMMARY
{
  "original_training_images": 1526,
  "traditional_augmented_images": 4578,
  "gan_synthetic_images": 1275,
  "original_missing": 0,
  "gan_missing": 0,
  "traditional_missing": 0,
  "local_cache": "/content/IITD_notebook10_cache",
  "note": "GAN labels are conditioning labels and are not verified identities."
}

Cell 3 complete. All verified training images are now local.


In [4]:

from pathlib import Path
from PIL import Image
import pandas as pd
import torch
import json

LOCAL_ROOT = Path("/content/IITD_notebook10_cache")

original_manifest = pd.read_csv(
    LOCAL_ROOT / "original_train_local_manifest.csv"
)
gan_manifest = pd.read_csv(
    LOCAL_ROOT / "gan_local_manifest.csv"
)

# Count extracted traditional images
traditional_files = [
    p for p in (LOCAL_ROOT / "traditional_train").rglob("*")
    if p.is_file() and p.suffix.lower() in {".png", ".jpg", ".jpeg", ".bmp"}
]

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("Training device: CPU")

print("\nDataset counts")
print("Original training:", len(original_manifest))
print("Traditional augmented:", len(traditional_files))
print("GAN synthetic:", len(gan_manifest))
print("Original classes:", original_manifest["label"].nunique())
print("Traditional classes:", len(set(
    p.parent.name for p in traditional_files
)))
print("GAN proposed labels:", gan_manifest["label"].nunique())

# Inspect sample image dimensions and modes
sample_paths = [
    Path(original_manifest.iloc[0]["local_path"]),
    traditional_files[0],
    Path(gan_manifest.iloc[0]["local_path"])
]

for path in sample_paths:
    with Image.open(path) as im:
        print(f"\n{path.name}: size={im.size}, mode={im.mode}")

print("\nCache directory:", LOCAL_ROOT)
print("Cache size (MB):", round(
    sum(p.stat().st_size for p in LOCAL_ROOT.rglob("*") if p.is_file())
    / (1024 ** 2), 2
))

print("\nCell 4 complete. No training started.")


PyTorch: 2.11.0+cpu
CUDA available: False
Training device: CPU

Dataset counts
Original training: 1526
Traditional augmented: 4578
GAN synthetic: 1275
Original classes: 425
Traditional classes: 425
GAN proposed labels: 425

01_L.png: size=(224, 224), mode=L

205_05_L_aug3.png: size=(224, 224), mode=L

label_000_sample_01.png: size=(224, 224), mode=RGB

Cache directory: /content/IITD_notebook10_cache
Cache size (MB): 242.0

Cell 4 complete. No training started.


In [5]:

# ============================================================
# NOTEBOOK 10: RECOGNITION WITH AUGMENTED DATA
# CPU-FRIENDLY END-TO-END EXPERIMENT
# ============================================================

from pathlib import Path
from PIL import Image
import json
import time
import shutil
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support
)
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. CPU SETTINGS
# ------------------------------------------------------------
SEED = 42
EPOCHS = 2             # Intentionally limited for CPU runtime
BATCH_SIZE = 8
NUM_WORKERS = 0
CPU_THREADS = 2        # Avoid using every CPU core

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(CPU_THREADS)

DEVICE = torch.device("cpu")
print("Device:", DEVICE)
print("CPU threads:", torch.get_num_threads())
print("Epochs per experiment:", EPOCHS)

# ------------------------------------------------------------
# 2. PATHS
# ------------------------------------------------------------
PROJECT = Path("/content/drive5/MyDrive/Ocular_Project")
CACHE = Path("/content/IITD_notebook10_cache")

PREPROCESS_DIR = (
    PROJECT / "results/IITD/notebook02_preprocessing/images_224"
)
SPLIT_CSV = (
    PROJECT / "results/IITD/notebook04_baseline_recognition/"
    "IITD_baseline_dataset_split.csv"
)
BASELINE_METRICS = (
    PROJECT / "results/IITD/notebook04_baseline_recognition/"
    "IITD_baseline_test_metrics.csv"
)
OUTPUT_DIR = PROJECT / "results/IITD/notebook10_recognition_augmented"
MODEL_DIR = PROJECT / "models/IITD"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# Save final results locally first; then copy to Drive.
LOCAL_OUTPUT = Path("/content/IITD_notebook10_results")
LOCAL_OUTPUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 3. LOAD SPLITS AND LOCAL MANIFESTS
# ------------------------------------------------------------
split_df = pd.read_csv(SPLIT_CSV)
original_manifest = pd.read_csv(
    CACHE / "original_train_local_manifest.csv"
)
gan_manifest = pd.read_csv(CACHE / "gan_local_manifest.csv")

required = {"split", "label", "label_name"}
if not required.issubset(split_df.columns):
    raise ValueError(
        f"Split CSV needs {required}; found {list(split_df.columns)}"
    )

if "original_relative_path" not in split_df.columns:
    raise ValueError(
        "Expected original_relative_path in the split CSV."
    )

train_df = split_df[split_df["split"] == "train"].copy()
val_df = split_df[split_df["split"] == "validation"].copy()
test_df = split_df[split_df["split"] == "test"].copy()

if len(train_df) != 1526 or len(val_df) != 218 or len(test_df) != 436:
    raise ValueError(
        f"Unexpected split sizes: train={len(train_df)}, "
        f"validation={len(val_df)}, test={len(test_df)}"
    )

# Map class names to the existing numeric labels.
class_map = (
    original_manifest[["label_name", "label"]]
    .drop_duplicates()
    .set_index("label_name")["label"]
    .to_dict()
)
NUM_CLASSES = len(class_map)

# ------------------------------------------------------------
# 4. PREPARE VALIDATION AND TEST IMAGES LOCALLY
# ------------------------------------------------------------
# They are copied only once. No validation/test image is used
# for gradient updates.

EVAL_CACHE = CACHE / "evaluation"
EVAL_CACHE.mkdir(parents=True, exist_ok=True)

def prepare_eval_records(frame, split_name):
    records = []
    split_dir = EVAL_CACHE / split_name
    split_dir.mkdir(parents=True, exist_ok=True)

    for row in tqdm(
        frame.itertuples(index=False),
        total=len(frame),
        desc=f"Preparing {split_name}"
    ):
        rel = Path(str(row.original_relative_path)).with_suffix(".png")
        source = PREPROCESS_DIR / rel

        if not source.exists():
            raise FileNotFoundError(f"Missing evaluation image: {source}")

        destination = split_dir / rel
        destination.parent.mkdir(parents=True, exist_ok=True)

        if not destination.exists():
            shutil.copy2(source, destination)

        records.append({
            "path": str(destination),
            "label": int(row.label)
        })

    return records

val_records = prepare_eval_records(val_df, "validation")
test_records = prepare_eval_records(test_df, "test")

# Original training records already point to local cache.
original_records = [
    {"path": str(row.local_path), "label": int(row.label)}
    for row in original_manifest.itertuples(index=False)
]

# Traditional augmented images are extracted under:
# traditional_train/train/<label_name>/<filename>
traditional_root = CACHE / "traditional_train" / "train"
traditional_records = []

for image_path in traditional_root.rglob("*"):
    if image_path.suffix.lower() not in {".png", ".jpg", ".jpeg", ".bmp"}:
        continue

    label_name = image_path.parent.name
    if label_name not in class_map:
        raise ValueError(f"Unknown traditional class: {label_name}")

    traditional_records.append({
        "path": str(image_path),
        "label": int(class_map[label_name])
    })

if len(original_records) != 1526:
    raise ValueError(f"Expected 1526 original images, got {len(original_records)}")
if len(traditional_records) != 4578:
    raise ValueError(
        f"Expected 4578 traditional images, got {len(traditional_records)}"
    )

# GAN labels are proposed conditioning labels, not verified identities.
gan_records = [
    {"path": str(row.local_path), "label": int(row.label)}
    for row in gan_manifest.itertuples(index=False)
]

if len(gan_records) != 1275:
    raise ValueError(f"Expected 1275 GAN images, got {len(gan_records)}")

print("\nData prepared:")
print("Classes:", NUM_CLASSES)
print("Original train:", len(original_records))
print("Traditional train augmentations:", len(traditional_records))
print("GAN synthetic images:", len(gan_records))
print("Validation:", len(val_records))
print("Test:", len(test_records))

# ------------------------------------------------------------
# 5. DATASET AND MODEL
# ------------------------------------------------------------
# Convert all images to grayscale and normalize consistently.
image_transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

class OcularDataset(Dataset):
    def __init__(self, records):
        self.records = records

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        item = self.records[index]
        with Image.open(item["path"]) as image:
            image = image.convert("RGB")
            image = image_transform(image)
        return image, int(item["label"])

def make_loader(records, shuffle):
    return DataLoader(
        OcularDataset(records),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=False
    )

val_loader = make_loader(val_records, shuffle=False)
test_loader = make_loader(test_records, shuffle=False)

def build_model():
    # No pretrained weights downloaded; avoids network dependency.
    model = models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(
        1, 64, kernel_size=7, stride=2, padding=3, bias=False
    )
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model.to(DEVICE)

# ------------------------------------------------------------
# 6. TRAIN AND EVALUATE ONE CONDITION
# ------------------------------------------------------------
def evaluate(model, loader):
    model.eval()
    all_true, all_pred = [], []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(DEVICE)
            logits = model(images)
            predictions = logits.argmax(dim=1).cpu().numpy()

            all_pred.extend(predictions.tolist())
            all_true.extend(labels.numpy().tolist())

    precision, recall, f1, _ = precision_recall_fscore_support(
        all_true,
        all_pred,
        labels=list(range(NUM_CLASSES)),
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": float(accuracy_score(all_true, all_pred)),
        "macro_precision": float(precision),
        "macro_recall": float(recall),
        "macro_f1": float(f1)
    }

def train_experiment(name, training_records, exploratory=False):
    print(f"\n{'=' * 65}")
    print(f"EXPERIMENT: {name}")
    print("Training images:", len(training_records))
    print("Exploratory / unverified labels:", exploratory)
    print(f"{'=' * 65}")

    loader = make_loader(training_records, shuffle=True)
    model = build_model()

    optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss()

    history = []
    best_val_accuracy = -1.0
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        start = time.time()
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        progress = tqdm(loader, desc=f"{name} epoch {epoch}/{EPOCHS}")

        for images, labels in progress:
            images = images.to(DEVICE)
            labels = labels.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            batch_size = labels.size(0)
            running_loss += loss.item() * batch_size
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += batch_size

            progress.set_postfix(loss=f"{loss.item():.3f}")

        train_loss = running_loss / max(total, 1)
        train_accuracy = correct / max(total, 1)
        val_metrics = evaluate(model, val_loader)
        elapsed = time.time() - start

        row = {
            "experiment": name,
            "epoch": epoch,
            "train_loss": train_loss,
            "train_accuracy": train_accuracy,
            "validation_accuracy": val_metrics["accuracy"],
            "validation_macro_f1": val_metrics["macro_f1"],
            "seconds": elapsed
        }
        history.append(row)

        print(
            f"Epoch {epoch}: loss={train_loss:.4f}, "
            f"train_acc={train_accuracy:.4f}, "
            f"val_acc={val_metrics['accuracy']:.4f}, "
            f"val_macro_f1={val_metrics['macro_f1']:.4f}, "
            f"time={elapsed / 60:.1f} min"
        )

        if val_metrics["accuracy"] > best_val_accuracy:
            best_val_accuracy = val_metrics["accuracy"]
            best_state = {
                key: value.detach().cpu().clone()
                for key, value in model.state_dict().items()
            }

        pd.DataFrame(history).to_csv(
            LOCAL_OUTPUT / f"{name}_training_history.csv", index=False
        )

    # Restore best validation checkpoint before final test evaluation.
    model.load_state_dict(best_state)
    test_metrics = evaluate(model, test_loader)

    checkpoint_path = LOCAL_OUTPUT / f"{name}_resnet18.pth"
    torch.save(model.state_dict(), checkpoint_path)

    result = {
        "experiment": name,
        "training_images": len(training_records),
        "epochs": EPOCHS,
        "best_validation_accuracy": best_val_accuracy,
        "test_images": len(test_records),
        **{f"test_{k}": v for k, v in test_metrics.items()},
        "exploratory_unverified_identity_labels": exploratory,
        "interpretation": (
            "Exploratory only: GAN conditioning labels are not verified "
            "biometric identities."
            if exploratory else
            "Closed-set experiment using the fixed original train/validation/test split."
        )
    }

    with open(LOCAL_OUTPUT / f"{name}_test_metrics.json", "w") as f:
        json.dump(result, f, indent=2)

    print("\nTest metrics:")
    print(json.dumps(result, indent=2))
    return result

# ------------------------------------------------------------
# 7. RUN EXPERIMENTS
# ------------------------------------------------------------
# This is deliberately limited to two epochs per new model.
# It may still take a substantial amount of time on CPU.

all_results = []

# A same-protocol original-only reference trained within this notebook.
all_results.append(
    train_experiment("original_only", original_records)
)

# Original data plus identity-preserving traditional transformations.
all_results.append(
    train_experiment(
        "original_plus_traditional",
        original_records + traditional_records
    )
)

# Exploratory only: synthetic GAN identity labels are unverified.
all_results.append(
    train_experiment(
        "original_plus_GAN_exploratory",
        original_records + gan_records,
        exploratory=True
    )
)

# ------------------------------------------------------------
# 8. COMPARISON TABLE AND DRIVE OUTPUTS
# ------------------------------------------------------------
comparison = pd.DataFrame(all_results)
comparison.to_csv(
    LOCAL_OUTPUT / "IITD_recognition_augmented_comparison.csv",
    index=False
)

# Include the previously recorded baseline metrics for reference.
# This does not retrain the baseline and preserves its original result.
if BASELINE_METRICS.exists():
    baseline_df = pd.read_csv(BASELINE_METRICS)
    baseline_df.to_csv(
        LOCAL_OUTPUT / "IITD_previous_baseline_reference.csv",
        index=False
    )

for file_path in LOCAL_OUTPUT.iterdir():
    if file_path.is_file():
        shutil.copy2(file_path, OUTPUT_DIR / file_path.name)

# Copy the final model checkpoints into the project model directory.
for file_path in LOCAL_OUTPUT.glob("*_resnet18.pth"):
    shutil.copy2(file_path, MODEL_DIR / f"IITD_notebook10_{file_path.name}")

print("\n" + "=" * 65)
print("NOTEBOOK 10 EXPERIMENTS COMPLETED")
print("=" * 65)
print(comparison.to_string(index=False))
print("\nResults saved to:", OUTPUT_DIR)
print("Model checkpoints saved to:", MODEL_DIR)
print("\nImportant:")
print("- Test set was held out from gradient updates.")
print("- GAN results are exploratory because identity labels are unverified.")
print("- Only two epochs were run; treat results as preliminary.")
print("- The earlier five-epoch baseline is a separate reference.")


Device: cpu
CPU threads: 2
Epochs per experiment: 2


Preparing validation:   0%|          | 0/218 [00:00<?, ?it/s]

Preparing test:   0%|          | 0/436 [00:00<?, ?it/s]


Data prepared:
Classes: 425
Original train: 1526
Traditional train augmentations: 4578
GAN synthetic images: 1275
Validation: 218
Test: 436

EXPERIMENT: original_only
Training images: 1526
Exploratory / unverified labels: False


original_only epoch 1/2:   0%|          | 0/191 [00:00<?, ?it/s]

...

Epoch 2: loss=4.8580, train_acc=0.0442, val_acc=0.0046, val_macro_f1=0.0002, time=29.9 min

Test metrics:
{
  "experiment": "original_plus_traditional",
  "training_images": 6104,
  "epochs": 2,
  "best_validation_accuracy": 0.013761467889908258,
  "test_images": 436,
  "test_accuracy": 0.022935779816513763,
  "test_macro_precision": 0.00429893658729363,
  "test_macro_recall": 0.02,
  "test_macro_f1": 0.005364338084061266,
  "exploratory_unverified_identity_labels": false,
  "interpretation": "Closed-set experiment using the fixed original train/validation/test split."
}

EXPERIMENT: original_plus_GAN_exploratory
Training images: 2801
Exploratory / unverified labels: True


original_plus_GAN_exploratory epoch 1/2:   0%|          | 0/351 [00:00<?, ?it/s]

Epoch 1: loss=6.2591, train_acc=0.0000, val_acc=0.0046, val_macro_f1=0.0000, time=14.0 min


original_plus_GAN_exploratory epoch 2/2:   0%|          | 0/351 [00:00<?, ?it/s]

Epoch 2: loss=6.0503, train_acc=0.0046, val_acc=0.0000, val_macro_f1=0.0000, time=13.8 min

Test metrics:
{
  "experiment": "original_plus_GAN_exploratory",
  "training_images": 2801,
  "epochs": 2,
  "best_validation_accuracy": 0.0045871559633027525,
  "test_images": 436,
  "test_accuracy": 0.0045871559633027525,
  "test_macro_precision": 1.3881658858233558e-05,
  "test_macro_recall": 0.002352941176470588,
  "test_macro_f1": 2.7600483008452648e-05,
  "exploratory_unverified_identity_labels": true,
  "interpretation": "Exploratory only: GAN conditioning labels are not verified biometric identities."
}

NOTEBOOK 10 EXPERIMENTS COMPLETED
                   experiment  training_images  epochs  best_validation_accuracy  test_images  test_accuracy  test_macro_precision  test_macro_recall  test_macro_f1  exploratory_unverified_identity_labels                                                                   interpretation
                original_only             1526       2                

In [7]:

from pathlib import Path
import json
import pandas as pd
import torch
import torch.nn as nn
from torchvision import models

PROJECT = Path("/content/drive5/MyDrive/Ocular_Project")

SPLIT_CSV = (
    PROJECT / "results/IITD/notebook04_baseline_recognition/"
    "IITD_baseline_dataset_split.csv"
)
ORIGINAL_MANIFEST = (
    Path("/content/IITD_notebook10_cache")
    / "original_train_local_manifest.csv"
)
TRAD_MANIFEST = (
    PROJECT / "results/IITD/notebook09_dataset_augmentation/"
    "IITD_traditional_train_manifest.csv"
)
GAN_MANIFEST = (
    PROJECT / "results/IITD/notebook09_dataset_augmentation/"
    "IITD_gan_manifest.csv"
)
BASELINE_CHECKPOINT = (
    PROJECT / "models/IITD/IITD_baseline_resnet18_best.pth"
)
BASELINE_METRICS = (
    PROJECT / "results/IITD/notebook04_baseline_recognition/"
    "IITD_baseline_test_metrics.csv"
)

print("=" * 60)
print("1. SPLIT AND LABEL CHECK")
print("=" * 60)

split = pd.read_csv(SPLIT_CSV)
original = pd.read_csv(ORIGINAL_MANIFEST)
traditional = pd.read_csv(TRAD_MANIFEST)
gan = pd.read_csv(GAN_MANIFEST)

print("Split columns:", split.columns.tolist())
print("Original manifest columns:", original.columns.tolist())
print("Traditional manifest columns:", traditional.columns.tolist())
print("GAN manifest columns:", gan.columns.tolist())

print("\nSplit sizes:")
print(split["split"].value_counts().to_dict())

print("\nClasses:")
print("Split classes:", split["label"].nunique())
print("Original manifest classes:", original["label"].nunique())
print("Traditional classes:", traditional["label"].nunique())
print("GAN proposed labels:", gan["label"].nunique())


# Check original training labels against the baseline split.
# Both CSVs use "relative_path" as the key.

split_train = split[split["split"] == "train"].copy()

merged = split_train.merge(
    original,
    on="relative_path",
    how="outer",
    suffixes=("_split", "_cache"),
    indicator=True
)

print("\nOriginal cache alignment:")
print("Merge counts:", merged["_merge"].value_counts().to_dict())

matched = merged[merged["_merge"] == "both"].copy()

label_match = (
    matched["label_split"].astype(int)
    == matched["label_cache"].astype(int)
)

print("Matched records:", len(matched))
print("Matching labels:", int(label_match.sum()))
print("Mismatched labels:", int((~label_match).sum()))

if (~label_match).any():
    print("\nMismatched examples:")
    print(
        matched.loc[
            ~label_match,
            ["relative_path", "label_split", "label_cache"]
        ].head(10).to_string(index=False)
    )

else:
    print("WARNING: original_relative_path missing from split CSV.")

print("\n" + "=" * 60)
print("2. CHECKPOINT CLASSIFIER DIMENSIONS")
print("=" * 60)

if BASELINE_CHECKPOINT.exists():
    try:
        # weights_only=True avoids loading arbitrary pickled objects.
        state = torch.load(
            BASELINE_CHECKPOINT,
            map_location="cpu",
            weights_only=True
        )

        print("Checkpoint type:", type(state).__name__)

        if isinstance(state, dict):
            print("Checkpoint keys (sample):", list(state.keys())[:8])

            # Handle either a raw state_dict or a common wrapper.
            if "state_dict" in state and isinstance(state["state_dict"], dict):
                state = state["state_dict"]
            elif "model_state_dict" in state and isinstance(
                state["model_state_dict"], dict
            ):
                state = state["model_state_dict"]

            for key in [
                "fc.weight", "fc.bias",
                "module.fc.weight", "module.fc.bias"
            ]:
                if key in state:
                    print(key, "shape:", tuple(state[key].shape))

            fc_keys = [
                (key, tuple(value.shape))
                for key, value in state.items()
                if key.endswith("fc.weight") or key.endswith("fc.bias")
            ]
            print("Classifier tensors:", fc_keys)

            # A 425-class model should have 425 output rows in fc.weight.
            weight_shapes = [
                shape for key, shape in fc_keys if key.endswith("fc.weight")
            ]
            if weight_shapes:
                print("Checkpoint classifier outputs:", weight_shapes[-1][0])
        else:
            print("Unexpected checkpoint structure; inspect manually.")

    except Exception as e:
        print("Could not inspect checkpoint:", repr(e))
else:
    print("Baseline checkpoint not found:", BASELINE_CHECKPOINT)

print("\n" + "=" * 60)
print("3. RECORDED BASELINE METRICS")
print("=" * 60)

if BASELINE_METRICS.exists():
    baseline_metrics = pd.read_csv(BASELINE_METRICS)
    print(baseline_metrics.to_string(index=False))
else:
    print("Baseline metrics CSV not found.")

print("\n" + "=" * 60)
print("4. NOTEBOOK 10 SAVED METRICS")
print("=" * 60)

notebook10_dir = (
    PROJECT / "results/IITD/notebook10_recognition_augmented"
)

for filename in [
    "original_only_test_metrics.json",
    "original_plus_traditional_test_metrics.json",
    "original_plus_GAN_exploratory_test_metrics.json"
]:
    path = notebook10_dir / filename
    if path.exists():
        with open(path) as f:
            result = json.load(f)
        print(f"\n{filename}")
        print(json.dumps(result, indent=2))
    else:
        print("Missing:", filename)

print("\nDiagnostic complete. No training performed.")


1. SPLIT AND LABEL CHECK
Split columns: ['image_path', 'relative_path', 'filename', 'subject_original', 'eye_original', 'capture_id_original', 'extension', 'valid_image', 'width_original', 'height_original', 'mode_original', 'format', 'file_size_bytes', 'original_relative_path', 'processed_path', 'subject_processed', 'eye_processed', 'capture_id_processed', 'width_processed', 'height_processed', 'mode_processed', 'path_exists', 'subject', 'eye', 'label_name', 'label', 'split']
Original manifest columns: ['relative_path', 'local_path', 'label', 'label_name']
Traditional manifest columns: ['original_relative_path', 'augmented_local_path', 'filename', 'subject', 'eye', 'label_name', 'label', 'split', 'augmentation_id', 'augmentation_methods', 'data_source', 'identity_label_verified']
GAN manifest columns: ['synthetic_path', 'filename', 'label', 'label_name', 'sample_index', 'source_model', 'native_resolution', 'saved_resolution', 'identity_preservation_verified', 'split', 'data_source', '

In [8]:

from pathlib import Path
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm

PROJECT = Path("/content/drive5/MyDrive/Ocular_Project")
CACHE = Path("/content/IITD_notebook10_cache")

SPLIT_CSV = (
    PROJECT / "results/IITD/notebook04_baseline_recognition/"
    "IITD_baseline_dataset_split.csv"
)
CHECKPOINT = PROJECT / "models/IITD/IITD_baseline_resnet18_best.pth"

split = pd.read_csv(SPLIT_CSV)
test_df = split[split["split"] == "test"].copy()

# Reproduce the same label ordering used by Notebook 04.
# The checkpoint stores class_names, so use those rather than
# assuming numeric labels are ordered correctly.
checkpoint = torch.load(
    CHECKPOINT, map_location="cpu", weights_only=True
)
class_names = checkpoint["class_names"]
state = checkpoint["model_state_dict"]

print("Checkpoint classes:", len(class_names))
print("Test images:", len(test_df))
print("Checkpoint metadata:", {
    k: v for k, v in checkpoint.items()
    if k != "model_state_dict"
})

# Reconstruct the label mapping from the checkpoint class order.
# Notebook 04 used label_name as its class identity.
class_to_idx = {name: i for i, name in enumerate(class_names)}

test_df["expected_label"] = test_df["label_name"].map(class_to_idx)

if test_df["expected_label"].isna().any():
    raise ValueError(
        "Some test label names do not occur in checkpoint class_names."
    )

test_df["expected_label"] = test_df["expected_label"].astype(int)

# Evaluation cache contains the same 224x224 preprocessed test images.
test_records = []
for row in test_df.itertuples(index=False):
    rel = Path(str(row.relative_path)).with_suffix(".png")
    path = CACHE / "evaluation" / "test" / rel

    if not path.exists():
        raise FileNotFoundError(f"Missing test image: {path}")

    test_records.append((str(path), int(row.expected_label)))

transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5])
])

class TestDataset(Dataset):
    def __len__(self):
        return len(test_records)

    def __getitem__(self, idx):
        path, label = test_records[idx]
        with Image.open(path) as img:
            x = transform(img.convert("RGB"))
        return x, label

loader = DataLoader(
    TestDataset(), batch_size=8, shuffle=False, num_workers=0
)

model = models.resnet18(weights=None)
model.conv1 = nn.Conv2d(
    1, 64, kernel_size=7, stride=2, padding=3, bias=False
)
model.fc = nn.Linear(model.fc.in_features, len(class_names))

model.load_state_dict(state)
model.eval()

predictions, actual = [], []

with torch.no_grad():
    for images, labels in tqdm(loader, desc="Baseline inference"):
        logits = model(images)
        predictions.extend(logits.argmax(1).tolist())
        actual.extend(labels.tolist())

accuracy = accuracy_score(actual, predictions)
print("\nBaseline checkpoint test accuracy:", round(accuracy, 4))
print("Baseline test accuracy (%):", round(accuracy * 100, 2))
print("Recorded Notebook 04 accuracy: 47.25%")


Checkpoint classes: 425
Test images: 436
Checkpoint metadata: {'num_classes': 425, 'class_names': ['001_L', '001_R', '002_L', '002_R', '003_L', '003_R', '004_L', '004_R', '005_L', '005_R', '006_L', '006_R', '007_L', '007_R', '008_L', '008_R', '009_L', '009_R', '010_L', '010_R', '011_L', '011_R', '012_L', '012_R', '013_L', '013_R', '014_L', '014_R', '015_L', '015_R', '016_L', '016_R', '017_L', '017_R', '018_L', '018_R', '019_L', '019_R', '020_L', '020_R', '021_L', '021_R', '022_L', '022_R', '023_L', '023_R', '024_L', '024_R', '025_L', '025_R', '026_L', '026_R', '027_L', '027_R', '028_L', '028_R', '029_L', '029_R', '030_L', '030_R', '031_L', '031_R', '032_L', '032_R', '033_L', '033_R', '034_L', '034_R', '035_L', '035_R', '036_L', '036_R', '037_L', '037_R', '038_L', '038_R', '039_L', '039_R', '040_L', '040_R', '041_L', '041_R', '042_L', '042_R', '043_L', '043_R', '044_L', '044_R', '045_L', '045_R', '046_L', '046_R', '047_L', '047_R', '048_L', '048_R', '049_L', '049_R', '050_L', '050_R', '

Baseline inference:   0%|          | 0/55 [00:00<?, ?it/s]


Baseline checkpoint test accuracy: 0.4725
Baseline test accuracy (%): 47.25
Recorded Notebook 04 accuracy: 47.25%


In [11]:

print("Merged columns:")
print(merged.columns.tolist())

# Find the actual class-name columns
name_cols = [c for c in merged.columns if "label_name" in c]
print("\nClass-name columns:", name_cols)

for col in name_cols:
    print(f"\n{col}:")
    print(merged[col].head().tolist())


Merged columns:
['image_path', 'relative_path', 'filename', 'subject_original', 'eye_original', 'capture_id_original', 'extension', 'valid_image', 'width_original', 'height_original', 'mode_original', 'format', 'file_size_bytes', 'original_relative_path', 'processed_path', 'subject_processed', 'eye_processed', 'capture_id_processed', 'width_processed', 'height_processed', 'mode_processed', 'path_exists', 'subject', 'eye', 'label_name_split', 'label_split', 'split', 'local_path', 'label_manifest', 'label_name_manifest']

Class-name columns: ['label_name_split', 'label_name_manifest']

label_name_split:
['001_L', '001_L', '001_L', '001_L', '001_R']

label_name_manifest:
['001_L', '001_L', '001_L', '001_L', '001_R']


In [12]:
from pathlib import Path

PROJECT = Path("/content/drive5/MyDrive/Ocular_Project")

paths = [
    "results/IITD/notebook10_recognition_augmented/"
    "IITD_recognition_augmented_comparison.csv",

    "results/IITD/notebook10_recognition_augmented/"
    "IITD_recognition_augmented_histories.csv",

    "results/IITD/notebook10_recognition_augmented/"
    "IITD_baseline_reference.csv",

    "results/IITD/notebook10_recognition_augmented/"
    "IITD_notebook10_original_only_metrics.json",

    "results/IITD/notebook10_recognition_augmented/"
    "IITD_notebook10_original_plus_traditional_metrics.json",

    "results/IITD/notebook10_recognition_augmented/"
    "IITD_notebook10_original_plus_GAN_exploratory_metrics.json",

    "models/IITD/IITD_notebook10_original_only_resnet18.pth",
    "models/IITD/IITD_notebook10_original_plus_traditional_resnet18.pth",
    "models/IITD/IITD_notebook10_original_plus_GAN_exploratory_resnet18.pth",
]

for relative_path in paths:
    path = PROJECT / relative_path
    status = f"{path.stat().st_size / (1024**2):.2f} MB" if path.is_file() else "MISSING"
    print(f"{'OK' if path.is_file() else 'MISSING'} | {relative_path} | {status}")

OK | results/IITD/notebook10_recognition_augmented/IITD_recognition_augmented_comparison.csv | 0.00 MB
MISSING | results/IITD/notebook10_recognition_augmented/IITD_recognition_augmented_histories.csv | MISSING
MISSING | results/IITD/notebook10_recognition_augmented/IITD_baseline_reference.csv | MISSING
MISSING | results/IITD/notebook10_recognition_augmented/IITD_notebook10_original_only_metrics.json | MISSING
MISSING | results/IITD/notebook10_recognition_augmented/IITD_notebook10_original_plus_traditional_metrics.json | MISSING
MISSING | results/IITD/notebook10_recognition_augmented/IITD_notebook10_original_plus_GAN_exploratory_metrics.json | MISSING
OK | models/IITD/IITD_notebook10_original_only_resnet18.pth | 43.52 MB
OK | models/IITD/IITD_notebook10_original_plus_traditional_resnet18.pth | 43.52 MB
OK | models/IITD/IITD_notebook10_original_plus_GAN_exploratory_resnet18.pth | 43.52 MB


In [13]:
from pathlib import Path

PROJECT = Path("/content/drive5/MyDrive/Ocular_Project")
OUT_DIR = PROJECT / "results/IITD/notebook10_recognition_augmented"

print("Notebook 10 output directory:", OUT_DIR)
print("Directory exists:", OUT_DIR.exists())

if OUT_DIR.exists():
    files = sorted(p for p in OUT_DIR.rglob("*") if p.is_file())

    print(f"\nTotal files found: {len(files)}")
    for p in files:
        print(f"{p.relative_to(OUT_DIR)}  |  {p.stat().st_size:,} bytes")
else:
    print("Check the active Google Drive mount path.")

Notebook 10 output directory: /content/drive5/MyDrive/Ocular_Project/results/IITD/notebook10_recognition_augmented
Directory exists: True

Total files found: 12
IITD_previous_baseline_reference.csv  |  275 bytes
IITD_recognition_augmented_comparison.csv  |  851 bytes
IITD_recognition_data_verification.json  |  305 bytes
original_only_resnet18.pth  |  45,632,587 bytes
original_only_test_metrics.json  |  478 bytes
original_only_training_history.csv  |  324 bytes
original_plus_GAN_exploratory_resnet18.pth  |  45,634,699 bytes
original_plus_GAN_exploratory_test_metrics.json  |  502 bytes
original_plus_GAN_exploratory_training_history.csv  |  305 bytes
original_plus_traditional_resnet18.pth  |  45,634,123 bytes
original_plus_traditional_test_metrics.json  |  471 bytes
original_plus_traditional_training_history.csv  |  350 bytes


In [14]:
from pathlib import Path
import json
import pandas as pd

PROJECT = Path("/content/drive5/MyDrive/Ocular_Project")
OUT = PROJECT / "results/IITD/notebook10_recognition_augmented"

json_files = [
    "original_only_test_metrics.json",
    "original_plus_traditional_test_metrics.json",
    "original_plus_GAN_exploratory_test_metrics.json",
    "IITD_recognition_data_verification.json",
]

csv_files = [
    "original_only_training_history.csv",
    "original_plus_traditional_training_history.csv",
    "original_plus_GAN_exploratory_training_history.csv",
    "IITD_recognition_augmented_comparison.csv",
    "IITD_previous_baseline_reference.csv",
]

for name in json_files:
    path = OUT / name
    with path.open() as f:
        data = json.load(f)
    print("VALID JSON:", name, "| keys:", list(data.keys()))

for name in csv_files:
    path = OUT / name
    df = pd.read_csv(path)
    print("VALID CSV:", name, "| rows:", len(df), "| columns:", list(df.columns))

print("\nNotebook 10 saved-results verification complete.")

VALID JSON: original_only_test_metrics.json | keys: ['experiment', 'training_images', 'epochs', 'best_validation_accuracy', 'test_images', 'test_accuracy', 'test_macro_precision', 'test_macro_recall', 'test_macro_f1', 'exploratory_unverified_identity_labels', 'interpretation']
VALID JSON: original_plus_traditional_test_metrics.json | keys: ['experiment', 'training_images', 'epochs', 'best_validation_accuracy', 'test_images', 'test_accuracy', 'test_macro_precision', 'test_macro_recall', 'test_macro_f1', 'exploratory_unverified_identity_labels', 'interpretation']
VALID JSON: original_plus_GAN_exploratory_test_metrics.json | keys: ['experiment', 'training_images', 'epochs', 'best_validation_accuracy', 'test_images', 'test_accuracy', 'test_macro_precision', 'test_macro_recall', 'test_macro_f1', 'exploratory_unverified_identity_labels', 'interpretation']
VALID JSON: IITD_recognition_data_verification.json | keys: ['original_train_expected', 'original_train_found', 'original_missing', 'tradi

## Conclusion — Recognition with Augmented Data

This notebook investigated ocular recognition using original training images, traditional augmentation, and exploratory GAN-generated images on the IIT Delhi Iris dataset.

### Experimental Results

| Training configuration | Training images | Test accuracy |
|---|---:|---:|
| Original only | 1,526 | 0.69% |
| Original + traditional augmentation | 6,104 | 2.29% |
| Original + exploratory GAN samples | 2,801 | 0.46% |

Each experiment was trained from scratch for only two epochs. The results were substantially below the 47.25% test accuracy obtained by the five-epoch baseline in Notebook 04.

### Interpretation and Limitations

- These preliminary experiments do not establish whether augmentation improves or degrades recognition performance.
- Training duration and experimental configuration differed from the baseline, so the results are not a controlled comparison.
- The GAN-generated images have not been verified to preserve the identities represented by their conditioning labels.
- The validation and test sets were kept separate from the augmented training data.
- Further controlled experiments would be required before drawing conclusions about the effectiveness of either augmentation method.

### Final Conclusion

The experiments demonstrate the implementation of an augmented-data recognition pipeline, but they do not demonstrate an improvement in recognition accuracy. The original baseline remains the reference result for this dataset. The augmented experiments are retained as preliminary findings, and no additional training was performed during finalization.

The next notebook compares the exploratory GAN and diffusion approaches, with their image-quality and identity-preservation limitations taken into account.